# Validating User Input and Data

Regular expressions are a powerful tool for pattern matching and searching in text. Beyond searching, they can also be used to **ensure data integrity** and **prevent code injection**. In this module we use Python's `re` module to validate usernames, numeric input, and email addresses, and to sanitize user input against dangerous characters.

## Overview

We will use Python Regex to do the following:

- Validate a username
- Validate numeric user input
- Validate email addresses and flag invalid entries
- Avoid code injection

## 1. Validating User Input

When users enter data into a form or CLI prompt, we need to make sure it meets our requirements before processing it. Python's `re.fullmatch()` function is ideal here — it requires the **entire** string to match the pattern, unlike `re.search()` which only needs a match anywhere in the string.

### 1.1 Validating a Username

**Requirements (typical example):**
- Only alphanumeric characters and underscores allowed
- Length between 3 and 20 characters

**Steps:**
1. Create a function that holds the pattern and uses `re.fullmatch()`
2. Ask for user input
3. Call the function and check if the input is valid

In [1]:
import re

def validate_username(username):
    """Validate that username contains only alphanumeric chars and underscores, 3–20 chars long."""
    pattern = r'^[a-zA-Z0-9_]{3,20}$'
    return re.fullmatch(pattern, username) is not None

# Test with various usernames
test_usernames = ["alice", "bob_99", "ab", "this_username_is_way_too_long_to_be_valid", "invalid user!", "valid_User123"]

for name in test_usernames:
    result = validate_username(name)
    print(f"{name!r:45} -> {'Valid' if result else 'Invalid'}")

'alice'                                       -> Valid
'bob_99'                                      -> Valid
'ab'                                          -> Invalid
'this_username_is_way_too_long_to_be_valid'   -> Invalid
'invalid user!'                               -> Invalid
'valid_User123'                               -> Valid


**Adjust the pattern to your requirements.** For example, to require the username to start with a letter:

```
pattern = r'^[a-zA-Z][a-zA-Z0-9_]{2,19}$'
```

## 2. Checking Numeric Data

Sometimes you want to make sure the user **only inputs numbers**. This can be done with regular expressions.

**Steps:**
1. Create a function that calls `re.search()` with the numeric pattern
2. Get user input
3. Check the user input with the validation function

In [2]:
import re

def is_numeric(value):
    """Return True if value contains only digits."""
    pattern = r'^\d+$'
    return re.search(pattern, value) is not None

# Test with various inputs
test_values = ["12345", "42", "3.14", "123abc", "0", "-7", ""]

for val in test_values:
    result = is_numeric(val)
    print(f"{val!r:15} -> {'Numeric' if result else 'Not numeric'}")

'12345'         -> Numeric
'42'            -> Numeric
'3.14'          -> Not numeric
'123abc'        -> Not numeric
'0'             -> Numeric
'-7'            -> Not numeric
''              -> Not numeric


**Adjust the examples for your requirements.** For example, to also accept negative integers and decimals:

```python
pattern = r'^-?\d+(\.\d+)?$'
```

## 3. Validating Email Addresses

Email addresses are a common part of many datasets and ensuring that they are valid is crucial to maintaining the **quality and integrity** of your data.

### Building the Email Pattern Step by Step

The email pattern is built up incrementally:

| Part | Pattern | Meaning |
|------|---------|----------|
| Local part | `^[a-zA-Z0-9._%+-]+` | One or more allowed characters before `@` |
| `@` sign | `@` | Literal `@` |
| Domain | `[a-zA-Z0-9.-]+` | Domain name |
| Dot before TLD | `\.` | Escaped literal `.` |
| TLD | `[a-zA-Z]{2,}$` | Top-level domain (at least 2 letters) |

Complete pattern: `^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$`

In [3]:
import re

def is_valid_email(email):
    """Validate an email address against a standard pattern."""
    pattern = r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$'
    return re.fullmatch(pattern, email) is not None

# Test with various email addresses
test_emails = [
    "user@example.com",
    "alice.bob+tag@domain.co.uk",
    "invalid-email",
    "missing@domain",
    "@nodomain.com",
    "spaces in@email.com",
    "valid_123@test.org",
]

for email in test_emails:
    result = is_valid_email(email)
    print(f"{email!r:35} -> {'Valid' if result else 'Invalid'}")

'user@example.com'                  -> Valid
'alice.bob+tag@domain.co.uk'        -> Valid
'invalid-email'                     -> Invalid
'missing@domain'                    -> Invalid
'@nodomain.com'                     -> Invalid
'spaces in@email.com'               -> Invalid
'valid_123@test.org'                -> Valid


### 3.1 Validate Email Addresses from a File and Flag Invalid Entries

A real-world use case: read a list of email addresses from a file, filter out invalid ones, and save them separately.

**Steps:**
1. Define the pattern
2. Read the list of email addresses from a file
3. Filter invalid email addresses
4. Save the invalid ones to a separate file

In [4]:
import re

EMAIL_PATTERN = re.compile(r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$')

# Simulated list of emails (in production, read from a file with open())
email_list = [
    "good@example.com",
    "also.good@company.org",
    "bad-email",
    "missing_at_sign.com",
    "another@valid.net",
    "broken@",
]

valid_emails = [e for e in email_list if EMAIL_PATTERN.fullmatch(e)]
invalid_emails = [e for e in email_list if not EMAIL_PATTERN.fullmatch(e)]

print("Valid emails:")
for e in valid_emails:
    print(f"  {e}")

print("\nInvalid emails (would be saved to file):")
for e in invalid_emails:
    print(f"  {e}")

# To save to file:
# with open('invalid_emails.txt', 'w') as f:
#     f.write('\n'.join(invalid_emails))

Valid emails:
  good@example.com
  also.good@company.org
  another@valid.net

Invalid emails (would be saved to file):
  bad-email
  missing_at_sign.com
  broken@


**Adjust to meet your requirements** — for example, restricting to a specific domain:

```python
pattern = r'^[a-zA-Z0-9._%+-]+@mycompany\.com$'
```

## 4. Avoiding Code Injection

**Code injection** happens when you directly insert user input into lines of code you are going to execute (e.g., SQL queries, shell commands, `eval()` calls). A malicious user can craft input that changes the logic of your program.

Use regex to:
- **Detect** dangerous characters or SQL keywords in input
- **Sanitize** input by removing or escaping those characters

> **Important:** Regex sanitization should **not be the only safeguard**. Always use parameterized queries for databases, and avoid `eval()` or `exec()` on user input whenever possible.

### 4.1 Sanitizing User Input Against Code Injection

**Steps:**
1. Create a function for sanitizing the user input
2. Create a function to check for dangerous characters
3. Sanitize the input and check it for code injection indicators

In [5]:
import re

# Characters commonly used in SQL injection and shell injection
DANGEROUS_PATTERN = re.compile(r"[;'\"\\<>|&`$]")  # semicolons, quotes, shell operators

# Common SQL injection keywords
SQL_KEYWORDS_PATTERN = re.compile(
    r'\b(SELECT|INSERT|UPDATE|DELETE|DROP|UNION|OR|AND|EXEC|EXECUTE)\b',
    re.IGNORECASE
)

def sanitize_input(user_input):
    """Remove dangerous characters from user input."""
    sanitized = DANGEROUS_PATTERN.sub('', user_input)
    return sanitized

def has_code_injection(user_input):
    """Return True if input contains dangerous characters or SQL keywords."""
    if DANGEROUS_PATTERN.search(user_input):
        return True
    if SQL_KEYWORDS_PATTERN.search(user_input):
        return True
    return False

# Test inputs
test_inputs = [
    "hello world",
    "alice'; DROP TABLE users; --",
    "SELECT * FROM accounts",
    "normal_user_123",
    "|rm -rf /",
]

for inp in test_inputs:
    sanitized = sanitize_input(inp)
    dangerous = has_code_injection(inp)
    print(f"Original : {inp!r}")
    print(f"Sanitized: {sanitized!r}")
    print(f"Dangerous: {dangerous}")
    print()

Original : 'hello world'
Sanitized: 'hello world'
Dangerous: False

Original : "alice'; DROP TABLE users; --"
Sanitized: 'alice DROP TABLE users --'
Dangerous: True

Original : 'SELECT * FROM accounts'
Sanitized: 'SELECT * FROM accounts'
Dangerous: True

Original : 'normal_user_123'
Sanitized: 'normal_user_123'
Dangerous: False

Original : '|rm -rf /'
Sanitized: 'rm -rf /'
Dangerous: True



### 4.2 A Complete Validation + Injection Check Pipeline

Combining all the techniques above into a single validation helper:

In [6]:
import re

def validate_and_sanitize(user_input, input_type='text'):
    """
    Validate and sanitize user input.
    input_type: 'text', 'username', 'numeric', or 'email'
    Returns (is_valid, sanitized_value, message)
    """
    # First check for injection
    dangerous = re.search(r"[;'\"\\<>|&`$]", user_input)
    sql_keywords = re.search(r'\b(SELECT|INSERT|UPDATE|DELETE|DROP|UNION)\b', user_input, re.IGNORECASE)
    
    if dangerous or sql_keywords:
        sanitized = re.sub(r"[;'\"\\<>|&`$]", '', user_input)
        sanitized = re.sub(r'\b(SELECT|INSERT|UPDATE|DELETE|DROP|UNION)\b', '', sanitized, flags=re.IGNORECASE)
        return False, sanitized, "Warning: potentially dangerous input detected and sanitized."

    patterns = {
        'username': r'^[a-zA-Z0-9_]{3,20}$',
        'numeric':  r'^-?\d+(\.\d+)?$',
        'email':    r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$',
        'text':     r'^[\w\s.,!?-]{1,200}$',
    }

    pattern = patterns.get(input_type, patterns['text'])
    if re.fullmatch(pattern, user_input):
        return True, user_input, "Input is valid."
    else:
        return False, user_input, f"Input does not match expected format for type '{input_type}'."


# Demo
examples = [
    ("alice_99",            "username"),
    ("bad user!",           "username"),
    ("42",                  "numeric"),
    ("3.14",                "numeric"),
    ("hello world",         "text"),
    ("user@example.com",    "email"),
    ("'; DROP TABLE--",     "text"),
]

for value, vtype in examples:
    valid, sanitized, msg = validate_and_sanitize(value, vtype)
    print(f"[{vtype:8}] {value!r:30} -> valid={valid}, msg={msg}")

[username] 'alice_99'                     -> valid=True, msg=Input is valid.
[username] 'bad user!'                    -> valid=False, msg=Input does not match expected format for type 'username'.
[numeric ] '42'                           -> valid=True, msg=Input is valid.
[numeric ] '3.14'                         -> valid=True, msg=Input is valid.
[text    ] 'hello world'                  -> valid=True, msg=Input is valid.
[email   ] 'user@example.com'             -> valid=True, msg=Input is valid.
[text    ] "'; DROP TABLE--"              -> valid=False, msg=Warning: potentially dangerous input detected and sanitized.


## Summary

| Topic | Key function | Key pattern example |
|---|---|---|
| Username validation | `re.fullmatch()` | `^[a-zA-Z0-9_]{3,20}$` |
| Numeric validation | `re.search()` | `^\d+$` |
| Email validation | `re.fullmatch()` | `^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$` |
| Injection detection | `re.search()` | `[;'"\\<>\|&\`$]` |
| Injection sanitization | `re.sub()` | Same pattern, replace with `''` |

**Key takeaways:**
- Use `re.fullmatch()` when the entire string must conform to a pattern.
- Use `re.search()` when you only need to detect the presence of a pattern anywhere.
- Regex sanitization is a useful layer of defense, but **should not be the only safeguard** — use parameterized queries, ORM frameworks, and principle of least privilege alongside it.

> Demo code reference: https://github.com/BrightBoost/pythonregex